# Notebook 02: scikit-learn ML Pipeline

> **Goal:** Build a production-quality scikit-learn pipeline that handles preprocessing, feature engineering, and modeling in a single reusable object.

**Dataset:** Titanic (binary classification)

**What you'll learn:**
1. Why pipelines matter
2. ColumnTransformer for mixed feature types
3. Custom transformers
4. Cross-validation the right way
5. Model comparison
6. Saving and loading pipelines

**Time:** ~3 hours

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay
)
import joblib
import warnings
warnings.filterwarnings('ignore')

print('Imports successful!')

## Part 1: Load and Prepare Data

In [ ]:
URL = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(URL)

# Define target and features
TARGET = 'Survived'
FEATURES = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Name']

X = df[FEATURES]
y = df[TARGET]

# Stratified split (preserves class balance in train/test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train)}')
print(f'Test size:  {len(X_test)}')
print(f'Train survival rate: {y_train.mean():.3f}')
print(f'Test survival rate:  {y_test.mean():.3f}')

## Part 2: Why Pipelines?

> **Key insight:** Fitting preprocessors on the full dataset before splitting is **data leakage**. Pipelines prevent this automatically — they fit only on training data.

In [ ]:
# BAD: This leaks information from test set into scaler
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)  # Uses full dataset!
# X_train_scaled, X_test_scaled = train_test_split(X_scaled, ...)
# Now test data influenced the scaler's mean/std — data leakage!

# GOOD: Pipeline ensures fit only on training data
# pipeline.fit(X_train, y_train)   # scaler fits on X_train only
# pipeline.score(X_test, y_test)   # scaler transforms X_test using training statistics

print('Pipeline approach eliminates data leakage automatically.')
print('This is especially critical for: StandardScaler, TargetEncoder, PCA, LDA')

## Part 3: Custom Feature Engineering Transformer

In [ ]:
class TitanicFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Custom sklearn transformer for Titanic feature engineering.
    Follows sklearn API: fit(), transform(), fit_transform()
    """

    # Titles that indicate higher social status
    RARE_TITLES = {'Dr', 'Rev', 'Col', 'Major', 'Mlle', 'Countess',
                   'Ms', 'Lady', 'Jonkheer', 'Don', 'Mme', 'Capt', 'Sir'}

    def fit(self, X, y=None):
        """Nothing to fit — this is a deterministic transformation."""
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Add engineered features."""
        X = X.copy()

        # Title extraction
        X['Title'] = X['Name'].str.extract(r',\s*(\w+)\.', expand=False)
        X['Title'] = X['Title'].apply(
            lambda t: t if t not in self.RARE_TITLES else 'Rare'
        )

        # Family features
        X['FamilySize'] = X['SibSp'] + X['Parch'] + 1
        X['IsAlone'] = (X['FamilySize'] == 1).astype(int)

        # Drop original Name column (no longer needed)
        X = X.drop('Name', axis=1)

        return X

    def get_feature_names_out(self, input_features=None):
        """Required for sklearn's set_output API."""
        return ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
                'Embarked', 'Title', 'FamilySize', 'IsAlone']


# Test the transformer
fe = TitanicFeatureEngineer()
X_engineered = fe.fit_transform(X_train)
print('After feature engineering:')
print(X_engineered.dtypes)
print(f'\nNew columns: {[c for c in X_engineered.columns if c not in X_train.columns]}')

## Part 4: Full Pipeline with ColumnTransformer

In [ ]:
# After feature engineering, these are our column types:
NUMERIC_FEATURES = ['Age', 'Fare', 'SibSp', 'Parch', 'FamilySize', 'IsAlone']
CATEGORICAL_FEATURES = ['Sex', 'Embarked', 'Title']
ORDINAL_FEATURES = ['Pclass']  # Ordered: 1 > 2 > 3 quality

# Numeric preprocessing: impute missing → scale to standard normal
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

# Categorical preprocessing: impute missing → one-hot encode
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# Ordinal preprocessing: impute only (tree models don't need scaling)
ordinal_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
])

# Combine all preprocessors
preprocessor = ColumnTransformer([
    ('numeric', numeric_transformer, NUMERIC_FEATURES),
    ('categorical', categorical_transformer, CATEGORICAL_FEATURES),
    ('ordinal', ordinal_transformer, ORDINAL_FEATURES),
])

# Full pipeline: feature engineering → preprocessing → model
pipeline = Pipeline([
    ('feature_engineering', TitanicFeatureEngineer()),
    ('preprocessing', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42)),
])

# Train
pipeline.fit(X_train, y_train)

# Evaluate
train_auc = roc_auc_score(y_train, pipeline.predict_proba(X_train)[:, 1])
test_auc = roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1])

print(f'Train AUC: {train_auc:.4f}')
print(f'Test AUC:  {test_auc:.4f}')
print(f'(Difference = {train_auc - test_auc:.4f} — higher difference = more overfitting)')

In [ ]:
# Detailed evaluation
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Did not survive', 'Survived']))

In [ ]:
# Confusion matrix + ROC curve side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Did Not Survive', 'Survived'],
    cmap='Blues',
    ax=axes[0]
)
axes[0].set_title('Confusion Matrix')

# ROC curve
y_proba = pipeline.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, 'b-', lw=2, label=f'Random Forest (AUC={test_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 5: Cross-Validation (The Right Way)

In [ ]:
# 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    pipeline,
    X_train, y_train,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1  # Use all CPU cores
)

print(f'CV AUC scores: {cv_scores.round(4)}')
print(f'Mean: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print()
print('Tip: The ± (std) tells you how stable the model is.')
print('A stable model has std < 0.02. A high std suggests high variance/overfitting.')

## Part 6: Model Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
}

results = {}
for name, model in models.items():
    pipe = Pipeline([
        ('feature_engineering', TitanicFeatureEngineer()),
        ('preprocessing', preprocessor),
        ('classifier', model),
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    results[name] = {'mean': scores.mean(), 'std': scores.std()}
    print(f'{name:25s}: {scores.mean():.4f} ± {scores.std():.4f}')

# Winner
best_model = max(results, key=lambda k: results[k]['mean'])
print(f'\nBest model: {best_model} (AUC = {results[best_model]["mean"]:.4f})')

In [ ]:
# Visualize comparison
names = list(results.keys())
means = [results[n]['mean'] for n in names]
stds = [results[n]['std'] for n in names]

plt.figure(figsize=(8, 4))
bars = plt.barh(names, means, xerr=stds, color='steelblue', alpha=0.8, capsize=5)
plt.axvline(x=0.8, color='red', linestyle='--', alpha=0.5, label='AUC=0.80 baseline')
plt.xlabel('AUC-ROC (5-fold CV)')
plt.title('Model Comparison — Titanic Survival Prediction')
plt.xlim(0.7, 1.0)
plt.legend()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## Part 7: Save and Load the Pipeline

In [ ]:
# Save the best pipeline
best_pipeline = Pipeline([
    ('feature_engineering', TitanicFeatureEngineer()),
    ('preprocessing', preprocessor),
    ('classifier', GradientBoostingClassifier(n_estimators=100, random_state=42)),
])
best_pipeline.fit(X_train, y_train)

# Save to disk
import os
os.makedirs('models', exist_ok=True)
joblib.dump(best_pipeline, 'models/titanic_pipeline_v1.pkl')
print('Pipeline saved!')

# Load and verify
loaded_pipeline = joblib.load('models/titanic_pipeline_v1.pkl')
loaded_auc = roc_auc_score(y_test, loaded_pipeline.predict_proba(X_test)[:, 1])
print(f'Loaded pipeline AUC: {loaded_auc:.4f}')

# Real-world prediction example
new_passenger = pd.DataFrame([{
    'Pclass': 1,
    'Sex': 'female',
    'Age': 29,
    'SibSp': 0,
    'Parch': 0,
    'Fare': 211.34,
    'Embarked': 'S',
    'Name': 'Graham, Miss. Margaret Edith',
}])

prob = loaded_pipeline.predict_proba(new_passenger)[0, 1]
print(f'\nNew passenger survival probability: {prob:.1%}')

## Summary

**Pipeline pattern you should always use:**
```
Pipeline([
    ('feature_engineering', CustomTransformer()),
    ('preprocessing', ColumnTransformer([...])),
    ('model', YourModel()),
])
```

**Why this matters in production:**
- No data leakage
- Single object to serialize
- Same transform applied at training and inference time
- Works with sklearn's cross_val_score, GridSearchCV, etc.

## Challenge
1. Add `FareBand` feature (use `pd.qcut`) to `TitanicFeatureEngineer`
2. Run `GridSearchCV` on the `GradientBoostingClassifier` parameters
3. Use `SHAP` to visualize feature importances for the best model